# AMPS II · Week 13 — Lab
## Porous media and conjugate heat transfer: Brinkman layers, a porous bed under free flow, a heated wall on submeshes

| | |
|---|---|
| **Lecture sections** | Part A (Darcy, Brinkman, Forchheimer) · Part B (one domain, the layers, the interface) · Part C (conjugate heat transfer on submeshes) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | write the exact channel solutions in NumPy · see a flow rate converge while the velocity does not · put an interface between mesh lines · solve flow on a submesh and heat on the whole domain, check $Nu=35/13$ and close the energy balance · judge an LLM's porous-layer script |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `porous.py` is written once with `%%writefile` (§1). All cells run with `-np 2`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A–C)*
`porous.py`: `channel(L, y0, y1, nx, ny)`, `tag_cells(msh, a)` (1 above $y=a$, 2 below), `Brinkman(msh, invK, G, mue, mu, beta)` (Taylor–Hood, one domain, `invK` a number or `invK_field(msh, ct, K)`),
`column`, `probe`; exact `brinkman_exact`, `two_layer_exact`, `ergun`; `Conjugate(msh, ct, ks, Pe, q, mode="single" | "contact", hc)` (Stokes on the fluid submesh, temperature on the parent mesh or on two
submeshes), `.bulk(x)`, `.column(x)`, `.balance()`, `developed_exact`. Run the cell.

In [ ]:
%%writefile /tmp/fnx/porous.py
"""porous.py — porous media (Brinkman, Brinkman-Forchheimer) and conjugate heat transfer on submeshes, DOLFINx 0.11 (AMPS II, Week 13)."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem, LinearProblem

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-11, snes_stol=0.0, snes_max_it=30)

def channel(L=4.0, y0=0.0, y1=1.0, nx=8, ny=16):
    return mesh.create_rectangle(comm, [[0, y0], [L, y1]], [nx, ny], mesh.CellType.quadrilateral)

def tag_cells(msh, a, lower=2, upper=1):
    """cell tags by the cell midpoint: y > a -> upper (1), y < a -> lower (2)"""
    tdim = msh.topology.dim; n = msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts
    mid = mesh.compute_midpoints(msh, tdim, np.arange(n, dtype=np.int32))
    return mesh.meshtags(msh, tdim, np.arange(n, dtype=np.int32), np.where(mid[:, 1] > a, upper, lower).astype(np.int32))

def probe(msh, f, P, comp=0):
    tdim = msh.topology.dim; nloc = msh.topology.index_map(tdim).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, tdim), P), P)
    out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
    for j in range(len(P)):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        val = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
        out[idx] = val[:, comp]; cnt[idx] = 1
    return comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

def column(msh, f, x, y0, y1, n=201, comp=0):
    P = np.zeros((n, 3)); P[:, 0] = x; P[:, 1] = np.linspace(y0, y1, n)
    return P[:, 1], probe(msh, f, P, comp)

# ═══════════════════════════════════════════════ Part 1: Brinkman / Brinkman-Forchheimer
class Brinkman:
    """-mu_e lap u + (mu/K) u + beta |u| u + grad p = G e_x, div u = 0 on a channel; walls y = y0, y1 no slip; ends do-nothing (p = 0)
    invK: a number or a DG0 Function (1/K, zero in free fluid); beta: Forchheimer coefficient (number or DG0)."""
    count = 0
    def __init__(self, msh, invK=0.0, G=1.0, mue=1.0, mu=1.0, beta=0.0):
        collect(); Brinkman.count += 1
        self.msh = msh; c = msh.basix_cell()
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        self.w = fem.Function(W); u, p = ufl.split(self.w); v, q = ufl.TestFunctions(W)
        un = ufl.sqrt(ufl.dot(u, u) + 1e-20)
        F = (mue * ufl.inner(ufl.grad(u), ufl.grad(v)) + mu * invK * ufl.dot(u, v) + beta * un * ufl.dot(u, v)
             - p * ufl.div(v) - q * ufl.div(u) - G * v[0]) * ufl.dx
        x = msh.geometry.x; y0 = comm.allreduce(x[:, 1].min(), op=MPI.MIN); y1 = comm.allreduce(x[:, 1].max(), op=MPI.MAX)
        V, _ = W.sub(0).collapse()
        walls = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], y0) | np.isclose(x[1], y1))
        self.bcs = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((W.sub(0), V), 1, walls), W.sub(0))]
        self.pb = NonlinearProblem(F, self.w, bcs=self.bcs, petsc_options_prefix=f"bk{Brinkman.count}_", petsc_options=NEWTON)
        self.y0, self.y1 = y0, y1; self.L = comm.allreduce(x[:, 0].max(), op=MPI.MAX)

    def solve(self):
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def u(self):
        return self.w.sub(0).collapse()

    def flow_rate(self, x=None):
        """volume flux through the section x (default: the middle) per unit depth"""
        y, ux = column(self.msh, self.u(), self.L / 2 if x is None else x, self.y0, self.y1, n=2001)
        return np.trapezoid(ux, y)

def invK_field(msh, ct, K, free_tag=1, porous_tag=2):
    """DG0 1/K: 0 in the free fluid, 1/K in the porous layer"""
    Q = fem.functionspace(msh, ("DG", 0)); f = fem.Function(Q)
    f.x.array[:] = 0.0; f.x.array[ct.find(porous_tag)] = 1.0 / K
    return f

def brinkman_exact(y, K, G=1.0, mu=1.0, mue=1.0):
    """porous channel 0 < y < 1 (walls no slip): u = G K/mu (1 - cosh((y - 1/2)/lam) / cosh(1/(2 lam))), lam = sqrt(mue K/mu)"""
    lam = np.sqrt(mue * K / mu)
    return G * K / mu * (1 - np.cosh((y - 0.5) / lam) / np.cosh(0.5 / lam))

def two_layer_exact(y, a, K, G=1.0, mu=1.0, mue=1.0):
    """porous layer 0 < y < a (Brinkman), free fluid a < y < 1 (Stokes), no slip at y = 0, 1; u and the shear stress continuous at y = a.
    Porous: u = ud + A exp(-y/lam) + B exp((y - a)/lam) (well scaled for thin layers); free: u = -G y^2/(2 mu) + C y + D."""
    lam = np.sqrt(mue * K / mu); ud = G * K / mu; e = np.exp(-a / lam)
    M = np.array([[1.0, e, 0.0, 0.0],                                        # u(0) = 0
                  [0.0, 0.0, 1.0, 1.0],                                      # u(1) = 0
                  [e, 1.0, -a, -1.0],                                        # continuity of u at a
                  [-mue / lam * e, mue / lam, -mu, 0.0]])                    # continuity of the shear stress at a
    rhs = np.array([-ud, G / (2 * mu), -ud - G / (2 * mu) * a ** 2, -G * a])
    A, B, C, D = np.linalg.solve(M, rhs)
    y = np.asarray(y, dtype=float); yp = np.minimum(y, a)
    return np.where(y < a, ud + A * np.exp(-yp / lam) + B * np.exp((yp - a) / lam), -G / (2 * mu) * y ** 2 + C * y + D)

def ergun(G, invK, beta, mu=1.0):
    """core (Darcy-Forchheimer) velocity: G = mu U / K + beta U^2"""
    if beta == 0:
        return G / (mu * invK)
    return (-mu * invK + np.sqrt((mu * invK) ** 2 + 4 * beta * G)) / (2 * beta)

# ═══════════════════════════════════════════════ Part 2: conjugate heat transfer
class Conjugate:
    """fluid channel 0 < y < 1 (tag 1) on a solid wall -t < y < 0 (tag 2), length L; heat flux q into the solid at y = -t, top y = 1 adiabatic.
    Flow: Stokes on the FLUID SUBMESH (parabolic inlet, do-nothing outlet), transferred to the parent mesh (zero in the solid).
    Temperature (nondimensional with H, mean velocity, fluid conductivity): Pe u . grad T = div(k grad T), k = 1 (fluid), ks (solid).
    mode "single": one P2 space on the parent mesh (T continuous); mode "contact": one space per submesh, coupled on the interface by a
    contact conductance hc: flux = hc (T_s - T_f) (hc -> infinity: perfect contact)."""
    count = 0
    def __init__(self, msh, ct, ks=10.0, Pe=20.0, q=1.0, mode="single", hc=None, flow="stokes", deg=2, rtol=1e-10):
        collect(); Conjugate.count += 1; cnt = Conjugate.count
        self.msh, self.ct = msh, ct; tdim = msh.topology.dim; fdim = tdim - 1
        x = msh.geometry.x; self.L = comm.allreduce(x[:, 0].max(), op=MPI.MAX); self.t = -comm.allreduce(x[:, 1].min(), op=MPI.MIN)
        self.Pe, self.ks, self.q = Pe, ks, q
        # ── flow on the fluid submesh ──
        self.fsub, self.fem_, _, _ = mesh.create_submesh(msh, tdim, ct.find(1))
        c = self.fsub.basix_cell()
        Wf = fem.functionspace(self.fsub, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        Vf, _ = Wf.sub(0).collapse()
        if flow == "stokes":
            (u, p), (v, qq) = ufl.TrialFunctions(Wf), ufl.TestFunctions(Wf)
            a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - qq * ufl.div(u)) * ufl.dx
            Lf = fem.Constant(self.fsub, 0.0) * qq * ufl.dx
            uin = fem.Function(Vf); uin.interpolate(lambda x: np.vstack([6 * x[1] * (1 - x[1]), 0 * x[1]]))
            inl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[0], 0))
            wl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[1], 0) | np.isclose(x[1], 1))
            bcs = [fem.dirichletbc(uin, fem.locate_dofs_topological((Wf.sub(0), Vf), fdim, inl), Wf.sub(0)),
                   fem.dirichletbc(fem.Function(Vf), fem.locate_dofs_topological((Wf.sub(0), Vf), fdim, wl), Wf.sub(0))]
            wh = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix=f"st{cnt}_", petsc_options=LU).solve()
            self.uf = wh.sub(0).collapse()
        else:
            self.uf = fem.Function(Vf); self.uf.interpolate(lambda x: np.vstack([6 * x[1] * (1 - x[1]), 0 * x[1]]))
        # transfer to the parent mesh: zero in the solid
        Vp = fem.functionspace(msh, ("Lagrange", 2, (2,))); self.u = fem.Function(Vp)
        nsub = self.fsub.topology.index_map(tdim).size_local + self.fsub.topology.index_map(tdim).num_ghosts
        sub_cells = np.arange(nsub, dtype=np.int32); parent_cells = self.fem_.sub_topology_to_topology(sub_cells, inverse=False)
        self.u.interpolate(self.uf, cells0=sub_cells, cells1=parent_cells)     # cells0: of the SOURCE (submesh), cells1: of the target
        self.u.x.scatter_forward()
        # ── temperature ──
        bot = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], -self.t))
        out = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.L))
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        tags = np.concatenate([bot, out, inl]); vals = np.r_[np.full(len(bot), 1), np.full(len(out), 2), np.full(len(inl), 3)].astype(np.int32)
        o = np.argsort(tags); self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, tags[o], vals[o]))
        self.dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
        self.mode = mode
        if mode == "single":
            S = fem.functionspace(msh, ("Lagrange", deg)); self.T = fem.Function(S); w = ufl.TestFunction(S)
            F = (Pe * ufl.dot(self.u, ufl.grad(self.T)) * w + ufl.dot(ufl.grad(self.T), ufl.grad(w))) * self.dx(1) \
                + ks * ufl.dot(ufl.grad(self.T), ufl.grad(w)) * self.dx(2) - q * w * self.ds(1)
            fin = np.intersect1d(inl, mesh.locate_entities(msh, fdim, lambda x: x[1] >= -1e-12))      # fluid part of the inlet: T = 0
            self.bcs = [fem.dirichletbc(fem.Function(S), fem.locate_dofs_topological(S, fdim, fin))]
            self.F_single = F
            self.pb = NonlinearProblem(F, self.T, bcs=self.bcs, petsc_options_prefix=f"ct{cnt}_", petsc_options=NEWTON)
        else:
            self.ssub, self.sem_, _, _ = mesh.create_submesh(msh, tdim, ct.find(2))
            Sf = fem.functionspace(self.fsub, ("Lagrange", deg)); Ss = fem.functionspace(self.ssub, ("Lagrange", deg))
            self.Tf, self.Ts = fem.Function(Sf), fem.Function(Ss); wf, ws = ufl.TestFunction(Sf), ufl.TestFunction(Ss)
            data = interface_data(msh, ct, 1, 2)
            dS = ufl.Measure("dS", domain=msh, subdomain_data=[(1, data)])(1)
            jump = self.Ts("-") - self.Tf("+")                                   # '+' = fluid
            Ff = (Pe * ufl.dot(self.u, ufl.grad(self.Tf)) * wf + ufl.dot(ufl.grad(self.Tf), ufl.grad(wf))) * self.dx(1) - hc * jump * wf("+") * dS
            Fs = ks * ufl.dot(ufl.grad(self.Ts), ufl.grad(ws)) * self.dx(2) + hc * jump * ws("-") * dS - q * ws * self.ds(1)
            finl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[0], 0))
            self.bcs = [fem.dirichletbc(fem.Function(Sf), fem.locate_dofs_topological(Sf, fdim, finl))]
            self.pb = NonlinearProblem([Ff, Fs], [self.Tf, self.Ts], bcs=self.bcs, entity_maps=[self.fem_, self.sem_], kind="mpi",
                                       petsc_options_prefix=f"cc{cnt}_", petsc_options=dict(NEWTON, snes_rtol=rtol))
            self.T = None

    def solve(self):
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def column(self, x, n=301):
        """y and T(x, y) through solid and fluid"""
        if self.mode == "single":
            return column(self.msh, self.T, x, -self.t, 1.0, n)
        ys, Ts = column(self.ssub, self.Ts, x, -self.t, 0.0, n // 3)
        yf, Tf = column(self.fsub, self.Tf, x, 0.0, 1.0, n)
        return np.r_[ys, yf], np.r_[Ts, Tf]

    def bulk(self, x, n=401):
        """mixing-cup temperature of the fluid at x (mean velocity 1)"""
        msh, T = (self.msh, self.T) if self.mode == "single" else (self.fsub, self.Tf)
        y, Tx = column(msh, T, x, 0.0, 1.0, n); _, ux = column(self.msh, self.u, x, 0.0, 1.0, n)
        return np.trapezoid(ux * Tx, y) / np.trapezoid(ux, y)

    def balance(self):
        """heat balance (per unit depth): heat in through the heated wall = convection out - convection in + conduction out through the ends.
        The end conduction uses the consistent (residual) flux at the Dirichlet inlet and the gradient elsewhere."""
        I = lambda e: comm.allreduce(fem.assemble_scalar(fem.form(e)), op=MPI.SUM)
        if self.mode != "single":
            raise ValueError("balance() is implemented for mode='single'")
        T = self.T; n = ufl.FacetNormal(self.msh); k = ufl.conditional(ufl.gt(ufl.SpatialCoordinate(self.msh)[1], 0), 1.0, self.ks)
        qin = I(self.q * self.ds(1))
        conv = I(self.Pe * self.u[0] * T * self.ds(2)) - I(self.Pe * self.u[0] * T * self.ds(3))
        cond = I(-k * ufl.dot(ufl.grad(T), n) * (self.ds(2) + self.ds(3)))
        # consistent flux at the Dirichlet inlet: the residual (assembled without boundary conditions) summed over the inlet dofs
        import dolfinx.fem.petsc as fp
        from petsc4py import PETSc
        R = fp.assemble_vector(fem.form(self.F_single)); R.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        dofs = self.bcs[0].dof_indices()[0]; nown = self.T.function_space.dofmap.index_map.size_local
        react = comm.allreduce(R.array[dofs[dofs < nown]].sum(), op=MPI.SUM); R.destroy()
        return dict(q_in=qin, convected=conv, conducted_ends_gradient=cond, inlet_consistent=-react, residual_gradient=qin - conv - cond, residual_consistent=qin - conv + react)

def interface_data(msh, ct, a=1, b=2):
    """owned interior facets between cells tagged a and b: [cell_a, local facet, cell_b, local facet, ...] ('+' = side a)"""
    tdim = msh.topology.dim; fdim = tdim - 1
    msh.topology.create_connectivity(fdim, tdim); msh.topology.create_connectivity(tdim, fdim)
    f2c, c2f = msh.topology.connectivity(fdim, tdim), msh.topology.connectivity(tdim, fdim)
    tag = np.zeros(msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts, dtype=np.int32)
    tag[ct.indices] = ct.values
    data = []
    for f in range(msh.topology.index_map(fdim).size_local):
        cs = f2c.links(f)
        if len(cs) == 2 and {tag[cs[0]], tag[cs[1]]} == {a, b}:
            c0, c1 = (cs[0], cs[1]) if tag[cs[0]] == a else (cs[1], cs[0])
            data += [c0, int(np.where(c2f.links(c0) == f)[0][0]), c1, int(np.where(c2f.links(c1) == f)[0][0])]
    return np.array(data, dtype=np.int32)

def developed_exact(y, t, ks, q=1.0, hc=np.inf):
    """fully developed temperature with uniform flux q (u = 6 y (1 - y), Pe dTb/dx = q), relative to the fluid-side interface value:
    fluid theta(y) = -q y + q (y^3 - y^4/2); solid theta = -q y / ks + q/hc (jump at the interface). Returns theta(y) and theta_bulk."""
    y = np.asarray(y, dtype=float)
    th = np.where(y >= 0, -q * y + q * (y ** 3 - y ** 4 / 2), -q * y / ks + (q / hc if np.isfinite(hc) else 0.0))
    yy = np.linspace(0, 1, 200001); thb = np.trapezoid(6 * yy * (1 - yy) * (-q * yy + q * (yy ** 3 - yy ** 4 / 2)), yy)
    return th, thb

## 2 · The exact solutions in NumPy  *(lecture §A2, §C2)*
Porous channel ($0<y<1$, no slip, $\mu=\mu_e=1$): $u=GK\big(1-\cosh\frac{y-1/2}{\sqrt K}/\cosh\frac1{2\sqrt K}\big)$, flow rate $Q=GK\big(1-2\sqrt K\tanh\frac1{2\sqrt K}\big)$.
Fully developed heating from one side (Poiseuille, other wall adiabatic): $\theta''=q\,u$, $\theta'(0)=-q$, $\theta'(1)=0$ → $Nu_H=q/(\theta_w-\theta_b)$.

In [ ]:
import numpy as np
for K in (1e-1, 1e-2, 1e-3, 1e-4):
    lam = np.sqrt(K); Q = K * (1 - 2 * lam * np.tanh(0.5 / lam))
    print(f"K = {K:g}: Brinkman layer sqrt(K) = {lam:.3f};  Q = {Q:.6e},  Darcy (plug) Q = {K:.1e},  Q / Q_Darcy = {Q / K:.4f}")
y = np.linspace(0, 1, 200001); u = 6 * y * (1 - y)
th = -y + y ** 3 - y ** 4 / 2                     # theta(y) - theta(0), q = 1
thb = np.trapezoid(u * th, y)
print(f"theta_bulk - theta_wall = {thb:.8f};  Nu_H = {-1 / thb:.6f} = 35/13 = {35 / 13:.6f};  Nu_Dh = {-2 / thb:.4f}")

> **✏️ Your turn (3 min).** (i) How many P2 cells must lie inside the Brinkman layer of $K=10^{-4}$ for a 1 % velocity, and how many cells across the channel does that mean? (ii) $Q/Q_\text{Darcy}$ is 0.98 at $K=10^{-4}$.
> What does that tell you about using the flow rate as a check of a Brinkman code?

> *your answer*

## 3 · The Brinkman layer: flow rate or velocity  *(lecture §B2)*
$K=10^{-2}$ and $10^{-4}$ on uniform meshes; the maximum velocity error and the flow-rate error against the exact solution.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from porous import *

for K in (1e-2, 1e-4):
    for ny in (8, 16, 32, 64):
        msh = channel(1.0, 0, 1, 2, ny); b = Brinkman(msh, invK=1 / K); r = b.solve()
        y, u = column(msh, b.u(), 0.5, 0, 1, 4001); ex = brinkman_exact(y, K)
        say(f"K {K:.0e}, {ny:3d} cells: max velocity error {np.abs(u - ex).max() / ex.max():.2e}, flow-rate error {abs(np.trapezoid(u, y) / np.trapezoid(ex, y) - 1):.2e}  (SNES {r})")

> **✏️ Your turn (4 min).** Read the orders of both columns for $K=10^{-4}$. Why does the flow rate converge faster than the velocity?

> *your answer*

## 4 · Free fluid over a porous layer: where is the interface?  *(lecture §A3, §B3)*
Porous layer $0<y<a$ ($K=10^{-3}$), free fluid above; one Brinkman equation with $1/K$ a DG0 field (zero in the fluid). Interface on mesh lines ($a=0.5$) and between them ($a=0.52$).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from porous import *

for a in (0.5, 0.52):
    for ny in (32, 64, 128):
        msh = channel(1.0, 0, 1, 2, ny); ct = tag_cells(msh, a); b = Brinkman(msh, invK=invK_field(msh, ct, 1e-3)); b.solve()
        y, u = column(msh, b.u(), 0.5, 0, 1, 4001); ex = two_layer_exact(y, a, 1e-3)
        say(f"a = {a}, {ny:3d} cells: max velocity error {np.abs(u - ex).max() / ex.max():.2e}, Q {np.trapezoid(u, y):.6f} (exact {np.trapezoid(ex, y):.6f})")

> **✏️ Your turn (5 min).** (i) Where does the one-domain model put the interface when $a$ lies inside a cell? (ii) Why is the error not decreasing smoothly for $a=0.52$?

> *your answer*

## 5 · Conjugate heat transfer: flow on a submesh, heat everywhere  *(lecture §C)*
Fluid channel $0<y<1$ over a solid wall $-0.5<y<0$ ($k_s=10$), $Pe=20$, heat flux $q=1$ into the bottom of the wall, $L=40$. Stokes is solved on the fluid submesh and interpolated onto the parent mesh;
the temperature lives on the whole mesh. At $x=20$ the flow is thermally developed.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from porous import *

msh = channel(40.0, -0.5, 1.0, 160, 30); ct = tag_cells(msh, 0.0)
cj = Conjugate(msh, ct, ks=10.0, Pe=20.0); r = cj.solve()
ys, Ts = column(msh, cj.T, 20.0, -0.5, -1e-9, 101); yf, Tf = column(msh, cj.T, 20.0, 0.0, 1.0, 201); Tb = cj.bulk(20.0)
ths, _ = developed_exact(np.minimum(ys, -1e-12), 0.5, 10.0); thf, thb = developed_exact(yf, 0.5, 10.0)
err = max(np.abs(Ts - Tf[0] - ths).max(), np.abs(Tf - Tf[0] - thf).max())
say(f"SNES {r}; dT_bulk/dx {(cj.bulk(22.0) - cj.bulk(18.0)) / 4:.6f} (q/Pe = 0.05); Nu_H {1 / (Tf[0] - Tb):.6f} (35/13 = {35 / 13:.6f}); max |theta - exact| {err:.1e}")
say("energy balance:", {k: round(float(v), 6) for k, v in cj.balance().items()})
cc = Conjugate(msh, ct, ks=10.0, Pe=20.0, mode="contact", hc=5.0); cc.solve()
_, a1 = column(cc.ssub, cc.Ts, 20.0, -1e-9, 0.0, 2); _, a2 = column(cc.fsub, cc.Tf, 20.0, 0.0, 1e-9, 2)
say(f"contact conductance hc = 5: temperature jump at the interface {a1[-1] - a2[0]:.6f} (q/hc = 0.2)")

> **✏️ Your turn (6 min).** (i) Where does the heat that is not convected out go? Why does the gradient-based balance leave 0.016, the consistent one round-off? (ii) Does the solid conductivity change $Nu$ in the developed region? Where does it matter?

> *your answer*

## 6 · LLM exercise: free fluid over a porous layer  *(lecture §D2)*
Ask an LLM for a DOLFINx 0.11 script for pressure-driven flow ($G=1$) in a channel $0<y<1$ whose lower half is a porous layer ($K=10^{-3}$, Brinkman with $\mu_e=\mu=1$), no slip at both walls, that prints the flow
rate and the velocity at the interface, **checked** against the exact two-layer solution. Paste it below, run it with `-np 2`, and judge it against §4.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line).

> *your verdict on the LLM's code*